# Triangle Shirtwaist Factory — Historical Generative Agent Simulation

**EDUC_5919 — Deep Learning and Transformer Models**

This notebook runs a three-agent generative simulation of the Triangle Shirtwaist Factory on the afternoon of **Saturday, March 25, 1911** — thirty minutes before the fire that killed 146 workers. The agents are:

- **Leonora Russo** (22) — Sicilian immigrant, shirtwaist cutter, supporting her family and saving for her sister's passage.
- **Max Bernstein** (48) — Jewish immigrant floor supervisor and partial owner, a self-made man.
- **Rosa Peretz** (24) — ILGWU organizer, survivor of the 1909 Uprising of the 20,000, banned from the building.

The goal is not to re-create the fire. It is to make visible the structural forces — economic desperation, identity and self-interest, legal permissiveness — that produced the catastrophe through many locally "rational" choices.

The architecture adapts the Stanford generative-agents design (Park et al., 2023) using the OpenAI chat-completions API. Each agent has an **identity stable set (ISS)**, **bootstrap memories**, and **constraints**, which are assembled into a system prompt per turn. The conversation runs as scripted turn-taking; each turn is a separate OpenAI API call.

## 1. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import os
from dataclasses import dataclass
from pathlib import Path

from openai import OpenAI

MODEL = "gpt-4o"
AGENTS_DIR = Path("agents")

assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY in your shell before launching Jupyter: "
    'export OPENAI_API_KEY="sk-..."'
)

## 2. The agent data model

Each agent JSON mirrors Stanford Town's `scratch.json` layering — `innate` (permanent traits), `learned` (stable background), `currently` (present goal) — plus a flat list of `bootstrap_memories` (what the agent knows, has seen, or fears) and `constraints` (explicit behavioral rules baked into the system prompt to keep responses historically grounded).

In [ ]:
@dataclass
class Agent:
    name: str
    first_name: str
    last_name: str
    age: int
    innate: str
    learned: str
    currently: str
    daily_plan_req: str
    bootstrap_memories: list
    constraints: list

def load_agent(path):
    with open(path) as f:
        return Agent(**json.load(f))

agents = {
    "leonora": load_agent(AGENTS_DIR / "leonora_russo.json"),
    "max":     load_agent(AGENTS_DIR / "max_bernstein.json"),
    "rosa":    load_agent(AGENTS_DIR / "rosa_peretz.json"),
}

for key, a in agents.items():
    print(f"{key:8s}  {a.name}, {a.age} — {a.innate}")

## 3. Building each agent's system prompt

`get_str_iss` is a direct port of Stanford Town's `Scratch.get_str_iss()` — the "bare minimum description of the persona that gets used in almost all prompts." `build_system_prompt` wraps the ISS with the scene description, memories, and constraints, and instructs the model to produce a single short in-character turn.

In [ ]:
def get_str_iss(agent):
    return (
        f"Name: {agent.name}\n"
        f"Age: {agent.age}\n"
        f"Innate traits: {agent.innate}\n"
        f"Learned traits: {agent.learned}\n"
        f"Currently: {agent.currently}\n"
        f"Daily plan requirement: {agent.daily_plan_req}\n"
        f"Current date: Saturday, March 25, 1911\n"
    )

def build_system_prompt(agent, scene_context):
    memories = "\n".join(f"- {m}" for m in agent.bootstrap_memories)
    constraints = "\n".join(f"- {c}" for c in agent.constraints)
    return f"""You are roleplaying a historical character in a classroom simulation designed to help students see how social, economic, and cultural forces shape individual choices. Stay in character. Do not narrate as a modern observer. Do not break the fourth wall.

# Scene
{scene_context}

# You are {agent.name}.
{get_str_iss(agent)}

# What is in your head (memories, knowledge, fears)
{memories}

# Constraints on your behavior
{constraints}

# How to respond
On each turn you will be shown the conversation so far. Respond with ONLY what {agent.first_name} says or does next — one to three sentences of dialogue, optionally with a brief physical action in *asterisks*. Do not include your name as a speaker label; the simulation adds that. Do not narrate other characters' thoughts or actions. Do not resolve the scene — stay in the moment."""

## 4. One turn = one OpenAI API call

For each agent turn we send:
- **system**: that agent's identity + scene + memories + constraints
- **user**: the running transcript of the conversation so far, and the instruction "what does \<name\> say next?"

This is the simplified Stanford Town pattern: each persona's turn is a separate LLM call, and the conversation history is passed as context so they respond to each other.

In [ ]:
def format_history(history):
    if not history:
        return "(The conversation has not begun yet. You are about to speak first.)"
    return "\n".join(f"{speaker}: {utterance}" for speaker, utterance in history)

def agent_turn(agent, history, scene_context, client):
    user_message = (
        f"Conversation so far on the 9th-floor cutting room:\n\n"
        f"{format_history(history)}\n\n"
        f"What does {agent.first_name} say or do next? Remember: one to three sentences, "
        f"in character, in period voice. No speaker label."
    )
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=300,
        messages=[
            {"role": "system", "content": build_system_prompt(agent, scene_context)},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content.strip()

## 5. The scene

The fixed setting: 9th floor cutting room, 4:30 PM, March 25, 1911. Rosa has slipped in through the freight entrance. Max is walking the floor. Leonora is at her cutting table. None of them knows what will happen in thirty minutes.

In [ ]:
SCENE_CONTEXT = """\
It is Saturday, March 25, 1911, at 4:30 PM, in New York City.

The scene is the 9th-floor cutting room of the Triangle Shirtwaist Company, in
the Asch Building at the corner of Washington Place and Greene Street. The floor
is loud with sewing machines on the 8th floor below and the hum of the cutting
tables here. Piles of cotton shirtwaist cuttings are heaped under the tables;
the air is thick with lint. The Washington Place stairwell door is locked, as
it is every shift; the Greene Street freight door is the only open exit. About
four hundred workers are on the floor, mostly young immigrant women. The shift
ends at 5:00 PM.

Leonora Russo is at her cutting table. Rosa Peretz, an ILGWU organizer banned
from the building by name, has slipped in through the freight entrance under a
shawl. She approaches Leonora. Max Bernstein, the floor supervisor and a partial
owner, is walking the floor and will shortly notice Rosa.

None of the three knows that in approximately thirty minutes, a scrap bin on
the 8th floor will ignite and the fire will spread through this building,
killing 146 workers. They are acting in the moment, with the knowledge and
fears available to each of them on this ordinary late-Saturday afternoon.
"""

## 6. Run the interaction

The turn order is scripted: Rosa approaches Leonora first, they exchange a few lines, Max notices Rosa and intervenes, and the three negotiate the confrontation. Each line is one OpenAI API call — expect the cell to take about 30–60 seconds.

In [ ]:
turn_order = [
    "rosa",      # Rosa opens: quiet approach, names the union and the ask
    "leonora",   # Leonora: hesitant, afraid
    "rosa",      # Rosa: specifics — locked doors, sprinklers
    "leonora",   # Leonora: the family, the sister
    "max",       # Max: recognizes Rosa, confronts
    "rosa",      # Rosa: holds ground
    "max",       # Max: the self-made-man frame, orders her out
    "leonora",   # Leonora: caught between, tries to disappear
    "rosa",      # Rosa: last word to Leonora
    "max",       # Max: closes the scene
]

client = OpenAI()
history = []

print("=" * 72)
print(" TRIANGLE SHIRTWAIST FACTORY — 9th FLOOR CUTTING ROOM")
print(" Saturday, March 25, 1911 — 4:30 PM")
print("=" * 72)
print()

for agent_key in turn_order:
    agent = agents[agent_key]
    utterance = agent_turn(agent, history, SCENE_CONTEXT, client)
    history.append((agent.first_name, utterance))
    print(f"{agent.first_name}: {utterance}")
    print()

## 7. What the transcript should reveal

The three-way conversation is designed to make structural forces visible through each agent's dialogue choices. When reading the transcript, look for:

- **Leonora's constraint is economic vulnerability.** She should hesitate, weigh the ask against her family's dependence on her wages, and fear the blacklist. Her agreement or refusal is not a free choice — it is shaped by her father's debt, her sister in Palermo, and her immigration status.
- **Max's constraint is the worldview that made him successful.** He should defend the factory sincerely — not as a villain, but as a man who believes what he says. His framing ("I came with fourteen dollars," "outsiders agitating my girls") is exactly how self-interest becomes invisible to the person holding it.
- **Rosa's constraint is legal and structural.** She should be specific (locked doors, sprinklers, sign the card), not rhetorical. Her urgency and the workers' hesitation together expose *why reform is hard even when danger is visible*.

The 146 deaths that occurred thirty minutes after this scene were not produced by a single villain. They were produced by every actor making choices that were locally rational within their constrained worlds. The point of the simulation is to make those constraints, and the system they compose, legible.